<a href="https://colab.research.google.com/github/kadimetla/AIML_Learning_Gang/blob/main/statistics/problem_notebooks/Problem02_House_Price_Prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>&nbsp;&nbsp;<a href="https://www.kaggle.com/kernels/welcome?src=https://raw.githubusercontent.com/kadimetla/AIML_Learning_Gang/main/statistics/problem_notebooks/Problem02_House_Price_Prediction.ipynb" target="_parent"><img src="https://kaggle.com/static/images/open-in-kaggle.svg" alt="Open In Kaggle"/></a>

**Compatible with Google Colab, Kaggle, JupyterLab, VS Code, and Binder.**

| Platform | How to open |
|----------|-------------|
| **Google Colab** | Click the Colab badge above |
| **Kaggle** | Click the Kaggle badge above, or *New Notebook → File → Import Notebook* |
| **JupyterLab** | `pip install jupyterlab && jupyter lab` in your terminal |
| **VS Code** | Open with the Jupyter extension (`.ipynb` support built-in) |
| **Binder** | Visit mybinder.org and point it to this repo |

> **Requirements:** `numpy` `pandas` `matplotlib` `seaborn` `scikit-learn` `scipy`  
> Pre-installed on Colab and Kaggle. The next cell installs them if missing.


In [ ]:
# Install required packages — safe on Colab, Kaggle, or local environments
%pip install numpy pandas matplotlib seaborn scikit-learn scipy --quiet


# 🏠 Problem 02: California House Price Prediction
## Statistics and ML Concepts Through a Real Problem

---

**The story:** We have data on 20,640 California neighborhoods (block groups) from the 1990 census. Each row describes a neighborhood: how wealthy residents are, how old the houses are, how many rooms, how many people — and what the median house price is.

**The question:** Can we understand *what drives* house prices — and predict the price of a neighborhood we've never seen?

**The approach:** Every statistical concept we introduce will answer a question the data raises. We don't learn statistics first and apply it later — we discover it as we need it.

---

| Section | What We Do |
|---------|------------|
| 0 | Setup |
| 1 | Meet the Problem |
| 2 | Data Health Check |
| 3 | Column-by-Column Analysis |
| 4 | Target Variable Deep Dive |
| 5 | Feature vs Target |
| 6 | Correlation Analysis |
| 7 | Feature Engineering |
| 8 | Building the Prediction |
| 9 | What We Learned |

## ⚙️ Section 0: Setup

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11

# Colormaps for price
PRICE_CMAP = 'plasma'
PRICE_CMAP2 = 'viridis'

print('All imports successful.')

## 🗺️ Section 1: Meet the Problem

### What are we predicting?
**Median house value** for a California block group — a small geographic unit containing ~400–3,000 people.

### Why it matters
- **Buyers** want to know if a listing is fairly priced
- **Sellers** want to set the right asking price
- **Investors** want to spot undervalued neighborhoods
- **City planners** need price models for zoning and policy

### What data do we have?
8 features describing each block group, recorded in the **1990 US Census**.

| Column | Plain English | Unit | Typical Range |
|--------|--------------|------|---------------|
| MedInc | Median household income | Tens of thousands USD | 0.5 – 15 |
| HouseAge | Median age of houses in block | Years | 1 – 52 |
| AveRooms | Avg rooms per household | Rooms | 1 – 10 (mostly) |
| AveBedrms | Avg bedrooms per household | Rooms | 0.5 – 2 (mostly) |
| Population | Total block group population | People | 3 – 35,682 |
| AveOccup | Avg household members | People | 1 – 10 (mostly) |
| Latitude | Geographic latitude | Degrees | 32.5 – 42.0 |
| Longitude | Geographic longitude | Degrees | -124.4 – -114.3 |
| **MedHouseValue** | **Median house value (TARGET)** | **$100,000s** | **0.15 – 5.0** |

In [ ]:
# Load the California Housing dataset
housing = fetch_california_housing()
df = pd.DataFrame(housing.data, columns=housing.feature_names)
df['MedHouseValue'] = housing.target  # target in $100k units

print(f'Dataset shape: {df.shape}')
print(f'Rows = {df.shape[0]:,} block groups | Columns = {df.shape[1]} (8 features + 1 target)')
print()

# Show first 10 rows
print('First 10 rows:')
df.head(10)

## 🩺 Section 2: Data Health Check

Before doing any analysis, we need to understand the *quality* of our data. Garbage in = garbage out.

**We check:**
1. Data types — are columns the right type?
2. Missing values — do we have gaps?
3. Basic descriptive statistics — what are the ranges, means, spreads?

In [ ]:
# Step 1: Data types and non-null counts
print('=== DATA TYPES AND NON-NULL COUNTS ===')
df.info()

In [ ]:
# Step 2: Descriptive statistics
print('=== DESCRIPTIVE STATISTICS ===')
desc = df.describe().T
desc['skewness'] = df.skew()
print(desc.round(3))
print()
print('What each stat means:')
print('  count  : number of non-null values')
print('  mean   : arithmetic average')
print('  std    : standard deviation — spread around the mean')
print('  min/max: smallest and largest values')
print('  25%/75%: first and third quartile (Q1, Q3)')
print('  50%    : median — middle value')
print('  skewness: >0 = right-skewed (tail to the right), <0 = left-skewed')

In [ ]:
# Step 3: Missing values
print('=== MISSING VALUES ===')
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
print(missing_df)
print()
print('Good news — California Housing has NO missing values. But always check!')

In [ ]:
# Visualize missing % per column (all zeros — good practice to show the process)
fig, ax = plt.subplots(figsize=(8, 4))
missing_pct.sort_values().plot(kind='barh', ax=ax, color='steelblue', edgecolor='white')
ax.set_xlabel('Missing %')
ax.set_title('Missing Values per Column (% of total rows)')
ax.axvline(x=0, color='red', linestyle='--', alpha=0.5, label='0% threshold')
ax.set_xlim(-0.5, 5)
ax.legend()
plt.tight_layout()
plt.show()
print('All columns have 0% missing — our dataset is complete.')

In [ ]:
# Distribution summary table
print('=== DISTRIBUTION SUMMARY ===')
summary = pd.DataFrame({
    'mean': df.mean(),
    'median': df.median(),
    'std': df.std(),
    'skewness': df.skew()
}).round(3)
print(summary)

## 🔍 Section 3: Column-by-Column Analysis

Let's understand each feature *before* we try to predict anything.

The goal: know your data like a domain expert. What's typical? What's strange? What might cause problems?

### 💰 Feature: MedInc (Median Income)

Median household income of the block group, in units of $10,000. A value of 3.0 means the median household earns ~$30,000/year.

In [ ]:
col = 'MedInc'
print(f'=== {col} STATISTICS ===')
print(f'  Mean:     {df[col].mean():.3f}')
print(f'  Median:   {df[col].median():.3f}')
print(f'  Std Dev:  {df[col].std():.3f}')
print(f'  Skewness: {df[col].skew():.3f}  (positive = right tail)')
print(f'  Min:      {df[col].min():.3f}')
print(f'  Max:      {df[col].max():.3f}')
print()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Histogram with KDE
axes[0].hist(df[col], bins=50, color='steelblue', edgecolor='white', density=True, alpha=0.7, label='Histogram')
kde_x = np.linspace(df[col].min(), df[col].max(), 300)
kde = stats.gaussian_kde(df[col])
axes[0].plot(kde_x, kde(kde_x), 'r-', lw=2, label='KDE')
axes[0].axvline(df[col].mean(), color='orange', lw=2, linestyle='--', label=f'Mean={df[col].mean():.2f}')
axes[0].axvline(df[col].median(), color='green', lw=2, linestyle=':', label=f'Median={df[col].median():.2f}')
axes[0].set_xlabel('MedInc ($10k)')
axes[0].set_ylabel('Density')
axes[0].set_title('MedInc Distribution')
axes[0].legend(fontsize=9)

# Box plot
axes[1].boxplot(df[col], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightblue', color='navy'),
                medianprops=dict(color='red', linewidth=2),
                flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[1].set_ylabel('MedInc ($10k)')
axes[1].set_title('MedInc Box Plot')
axes[1].set_xticks([])

# Empirical CDF
sorted_vals = np.sort(df[col])
cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
axes[2].plot(sorted_vals, cdf, 'steelblue', lw=2)
axes[2].axhline(0.5, color='red', linestyle='--', alpha=0.7, label='50th percentile')
axes[2].set_xlabel('MedInc ($10k)')
axes[2].set_ylabel('Cumulative Probability')
axes[2].set_title('Empirical CDF of MedInc')
axes[2].legend()

plt.suptitle('MedInc Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**📊 CONCEPT: Skewness and the Mean vs Median**

Income distributions are almost always **right-skewed** — a few very wealthy neighborhoods pull the mean above the median.

- **Skewness > 0**: right tail is longer (mean > median)
- **Skewness = 0**: perfectly symmetric (mean ≈ median)
- **Skewness < 0**: left tail is longer (mean < median)

When data is skewed, the **median** is a more robust measure of center than the mean — it's not pulled by extreme values.

**Insight:** Most block groups earn \$20k–\$50k/year median income. A small number of very wealthy neighborhoods extend the tail to \$150k+.

### 🏚️ Feature: HouseAge (Median House Age)

In [ ]:
col = 'HouseAge'
print(f'=== {col} STATISTICS ===')
print(f'  Mean:     {df[col].mean():.3f} years')
print(f'  Median:   {df[col].median():.3f} years')
print(f'  Std Dev:  {df[col].std():.3f}')
print(f'  Skewness: {df[col].skew():.3f}')
print(f'  Min/Max:  {df[col].min()} / {df[col].max()} years')
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df[col], bins=52, color='coral', edgecolor='white', alpha=0.8)
axes[0].axvline(df[col].mean(), color='blue', lw=2, linestyle='--', label=f'Mean={df[col].mean():.1f}')
axes[0].axvline(df[col].median(), color='green', lw=2, linestyle=':', label=f'Median={df[col].median():.1f}')
axes[0].set_xlabel('House Age (years)')
axes[0].set_ylabel('Count')
axes[0].set_title('HouseAge Distribution')
axes[0].legend()

axes[1].boxplot(df[col], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightsalmon', color='darkred'),
                medianprops=dict(color='blue', linewidth=2),
                flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[1].set_ylabel('House Age (years)')
axes[1].set_title('HouseAge Box Plot')
axes[1].set_xticks([])

plt.suptitle('HouseAge Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Note: There is a spike at 52 years — this is the dataset cap.')
print(f'Number of block groups with HouseAge == 52 (cap): {(df[col] == 52).sum():,}')

### 🛋️ Feature: AveRooms (Average Rooms per Household)

In [ ]:
col = 'AveRooms'
print(f'=== {col} STATISTICS ===')
print(f'  Mean:     {df[col].mean():.3f}')
print(f'  Median:   {df[col].median():.3f}')
print(f'  Std Dev:  {df[col].std():.3f}')
print(f'  Skewness: {df[col].skew():.3f}')
print()

# IQR method to find outliers
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
print(f'  Q1={Q1:.2f}, Q3={Q3:.2f}, IQR={IQR:.2f}')
print(f'  Outlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]')
print(f'  Number of outliers (IQR method): {len(outliers):,} ({len(outliers)/len(df)*100:.1f}%)')
print(f'  Max rooms in dataset: {df[col].max():.1f}')
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram with log scale on x (due to outliers)
axes[0].hist(np.log1p(df[col]), bins=60, color='mediumpurple', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('log(1 + AveRooms)')
axes[0].set_ylabel('Count')
axes[0].set_title('AveRooms Distribution (log scale)')

# Box plot
axes[1].boxplot(df[col], vert=True, patch_artist=True,
                boxprops=dict(facecolor='plum', color='purple'),
                medianprops=dict(color='red', linewidth=2),
                flierprops=dict(marker='.', markersize=2, alpha=0.2))
axes[1].set_ylabel('AveRooms')
axes[1].set_title('AveRooms Box Plot (outliers visible)')
axes[1].set_xticks([])

plt.suptitle('AveRooms Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**📊 CONCEPT: Outliers and the IQR Method**

An **outlier** is a value far from the main cluster of data. They can be genuine extreme cases, measurement errors, or data entry mistakes.

The **IQR (Interquartile Range) method** flags outliers as:
- Below `Q1 - 1.5 × IQR`
- Above `Q3 + 1.5 × IQR`

Where IQR = Q3 - Q1 (the middle 50% of the data).

**For AveRooms:** Most homes have 4–6 rooms, but some block groups have 100+ average rooms — these are likely errors or unusual communal housing.

### 🛏️ Feature: AveBedrms (Average Bedrooms per Household)

In [ ]:
col = 'AveBedrms'
print(f'=== {col} STATISTICS ===')
print(f'  Mean:     {df[col].mean():.3f}')
print(f'  Median:   {df[col].median():.3f}')
print(f'  Std Dev:  {df[col].std():.3f}')
print(f'  Skewness: {df[col].skew():.3f}')
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df[col] < Q1 - 1.5*IQR) | (df[col] > Q3 + 1.5*IQR)]
print(f'  Outliers (IQR): {len(outliers):,}')
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df[col].clip(upper=4), bins=50, color='teal', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('AveBedrms')
axes[0].set_ylabel('Count')
axes[0].set_title('AveBedrms Distribution (clipped at 4)')
axes[0].axvline(df[col].median(), color='red', lw=2, label=f'Median={df[col].median():.2f}')
axes[0].legend()

axes[1].boxplot(df[col], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightcyan', color='teal'),
                medianprops=dict(color='red', linewidth=2),
                flierprops=dict(marker='.', markersize=2, alpha=0.2))
axes[1].set_ylabel('AveBedrms')
axes[1].set_title('AveBedrms Box Plot')
axes[1].set_xticks([])

plt.suptitle('AveBedrms Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Most block groups average 0.8–1.5 bedrooms per household.')
print('Outliers exist — communal housing, unusual properties, or data errors.')

### 👥 Feature: Population

Total population of the block group. Highly right-skewed — perfect case for log transformation.

In [ ]:
col = 'Population'
print(f'=== {col} STATISTICS ===')
print(f'  Mean:     {df[col].mean():.1f}')
print(f'  Median:   {df[col].median():.1f}')
print(f'  Std Dev:  {df[col].std():.1f}')
print(f'  Skewness (raw):    {df[col].skew():.3f}')
log_pop = np.log1p(df[col])
print(f'  Skewness (log+1):  {log_pop.skew():.3f}')
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df[col], bins=80, color='tomato', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('Population')
axes[0].set_ylabel('Count')
axes[0].set_title(f'Population (raw)\nSkewness = {df[col].skew():.2f}')

axes[1].hist(log_pop, bins=60, color='darkorange', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('log(1 + Population)')
axes[1].set_ylabel('Count')
axes[1].set_title(f'Population (log transformed)\nSkewness = {log_pop.skew():.2f}')

plt.suptitle('Population: Before vs After Log Transform', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**📊 CONCEPT: Log Transformation**

When data is extremely right-skewed, a **log transformation** compresses the large values and spreads out the small ones — making the distribution more symmetric.

We use `log(1 + x)` (not `log(x)`) to handle zeros safely.

**Why it matters for ML:** Many algorithms (especially linear ones) assume features are roughly normally distributed. Skewed features can hurt model performance.

### 🏘️ Feature: AveOccup (Average Household Occupancy)

In [ ]:
col = 'AveOccup'
print(f'=== {col} STATISTICS ===')
print(f'  Mean:     {df[col].mean():.3f}')
print(f'  Median:   {df[col].median():.3f}')
print(f'  Std Dev:  {df[col].std():.3f}')
print(f'  Skewness: {df[col].skew():.3f}')
print(f'  Max:      {df[col].max():.1f}')
extreme = df[df[col] > 10]
print(f'  Block groups with AveOccup > 10: {len(extreme)} ({len(extreme)/len(df)*100:.2f}%)')
print()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Raw histogram
axes[0].hist(df[col], bins=100, color='seagreen', edgecolor='white', alpha=0.8)
axes[0].set_xlabel('AveOccup')
axes[0].set_ylabel('Count')
axes[0].set_title('AveOccup (raw — extreme outliers visible)')

# Without extreme outliers
clean = df[df[col] <= 10][col]
axes[1].hist(clean, bins=50, color='mediumseagreen', edgecolor='white', alpha=0.8)
axes[1].axvline(clean.mean(), color='blue', lw=2, linestyle='--', label=f'Mean={clean.mean():.2f}')
axes[1].axvline(clean.median(), color='red', lw=2, linestyle=':', label=f'Median={clean.median():.2f}')
axes[1].set_xlabel('AveOccup')
axes[1].set_ylabel('Count')
axes[1].set_title('AveOccup (outliers removed, ≤10)')
axes[1].legend()

# Box plot
axes[2].boxplot(df[col], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightgreen', color='darkgreen'),
                medianprops=dict(color='red', linewidth=2),
                flierprops=dict(marker='.', markersize=1, alpha=0.2))
axes[2].set_ylabel('AveOccup')
axes[2].set_title('AveOccup Box Plot')
axes[2].set_xticks([])

plt.suptitle('AveOccup Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Most households have 2–4 occupants. Extreme outliers (>10) may be data errors or unusual group housing.')

### 🌍 Features: Latitude & Longitude (Geography)

These aren't typical numeric features — they encode **location**. A scatter plot with color reveals spatial patterns no histogram can show.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 10))

sc = ax.scatter(
    df['Longitude'], df['Latitude'],
    c=df['MedHouseValue'], cmap=PRICE_CMAP,
    s=1.5, alpha=0.6, vmin=0, vmax=5
)
cbar = plt.colorbar(sc, ax=ax, pad=0.02)
cbar.set_label('Median House Value ($100k)', fontsize=11)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('California Block Groups: Price by Location\n(dark = cheap, yellow = expensive)', fontsize=13)

# Annotate key areas
ax.annotate('Bay Area\n(SF/Oakland)', xy=(-122.4, 37.8), fontsize=9,
            xytext=(-121.0, 38.5), arrowprops=dict(arrowstyle='->', color='white'), color='white')
ax.annotate('LA Basin', xy=(-118.3, 34.0), fontsize=9,
            xytext=(-116.8, 34.5), arrowprops=dict(arrowstyle='->', color='white'), color='white')
ax.annotate('San Diego', xy=(-117.2, 32.7), fontsize=9,
            xytext=(-115.8, 32.5), arrowprops=dict(arrowstyle='->', color='white'), color='white')

ax.set_facecolor('#1a1a2e')
fig.patch.set_facecolor('#1a1a2e')
ax.tick_params(colors='white')
ax.xaxis.label.set_color('white')
ax.yaxis.label.set_color('white')
ax.title.set_color('white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

plt.tight_layout()
plt.show()

print('Coastal areas — Bay Area, LA Basin, San Diego — show clusters of high prices (yellow/orange).')
print('Inland areas (Central Valley, Mojave) are predominantly low-price (dark).')

**📊 CONCEPT: Spatial Data Visualization**

When features are geographic coordinates, a **scatter plot with a color dimension** reveals patterns that statistics alone can't show.

This plot immediately tells us: **location is one of the strongest drivers of house price**. Coastal proximity, specifically the Bay Area and LA Basin corridors, commands a significant premium.

This is a pattern that a correlation matrix *cannot* capture — it's spatial, not linear.

### 🎯 Feature: MedHouseValue (Target Variable — Preview)

The value we're trying to predict. Units: $100,000s. So 2.5 = $250,000.

In [ ]:
col = 'MedHouseValue'
print(f'=== {col} STATISTICS ===')
print(f'  Mean:     ${df[col].mean()*100:.1f}k')
print(f'  Median:   ${df[col].median()*100:.1f}k')
print(f'  Std Dev:  ${df[col].std()*100:.1f}k')
print(f'  Skewness: {df[col].skew():.3f}')
print(f'  Min:      ${df[col].min()*100:.1f}k')
print(f'  Max:      ${df[col].max()*100:.1f}k (data cap at $500k)')
capped = (df[col] == 5.0).sum()
print(f'  Block groups capped at $500k: {capped:,} ({capped/len(df)*100:.1f}%)')
print()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram with KDE
axes[0].hist(df[col], bins=60, color='gold', edgecolor='white', density=True, alpha=0.7)
kde_x = np.linspace(df[col].min(), df[col].max(), 300)
kde = stats.gaussian_kde(df[col])
axes[0].plot(kde_x, kde(kde_x), 'r-', lw=2, label='KDE')
axes[0].axvline(df[col].mean(), color='blue', lw=2, linestyle='--', label=f'Mean=${df[col].mean()*100:.0f}k')
axes[0].axvline(df[col].median(), color='green', lw=2, linestyle=':', label=f'Median=${df[col].median()*100:.0f}k')
axes[0].axvline(5.0, color='red', lw=2, linestyle='-', alpha=0.5, label='Cap at $500k')
axes[0].set_xlabel('Median House Value ($100k)')
axes[0].set_ylabel('Density')
axes[0].set_title('MedHouseValue Distribution')
axes[0].legend(fontsize=9)

# Box plot
axes[1].boxplot(df[col], vert=True, patch_artist=True,
                boxprops=dict(facecolor='lightyellow', color='goldenrod'),
                medianprops=dict(color='red', linewidth=2),
                flierprops=dict(marker='.', markersize=2, alpha=0.3))
axes[1].set_ylabel('Median House Value ($100k)')
axes[1].set_title('MedHouseValue Box Plot')
axes[1].set_xticks([])

plt.suptitle('Target Variable: MedHouseValue', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🎯 Section 4: The Target Variable — Deep Dive

Before building any model, we need to deeply understand **what we're predicting**.

Key question: *What does the price distribution look like, and what does that tell us about how we should model it?*

In [ ]:
# Percentile analysis of house prices
percentiles = [10, 25, 50, 75, 90, 95, 99]
pct_vals = np.percentile(df['MedHouseValue'], percentiles)

print('=== PRICE PERCENTILES ===')
for p, v in zip(percentiles, pct_vals):
    print(f'  {p:3d}th percentile: ${v*100:.0f}k')

print()
print('Interpretation:')
print(f'  A $200k home is around the {np.searchsorted(pct_vals, 2.0)}0th percentile — typical CA home.')
print(f'  A $450k home is in approximately the top 10%.')
print(f'  $500k is the data cap — {(df["MedHouseValue"]==5.0).sum():,} block groups are capped here.')

In [ ]:
# Compare raw vs log-transformed price distribution
log_price = np.log1p(df['MedHouseValue'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Raw distribution with percentiles marked
axes[0].hist(df['MedHouseValue'], bins=60, color='gold', edgecolor='white', density=True, alpha=0.7)
for p, v in zip([25, 50, 75], np.percentile(df['MedHouseValue'], [25, 50, 75])):
    axes[0].axvline(v, lw=1.5, linestyle='--', label=f'P{p}=${v*100:.0f}k')
axes[0].set_xlabel('Median House Value ($100k)')
axes[0].set_ylabel('Density')
axes[0].set_title(f'Raw Price\n(skewness={df["MedHouseValue"].skew():.2f})')
axes[0].legend(fontsize=9)

# Log-transformed
axes[1].hist(log_price, bins=60, color='darkorange', edgecolor='white', density=True, alpha=0.7)
axes[1].axvline(log_price.mean(), color='blue', lw=2, linestyle='--', label=f'Mean={log_price.mean():.2f}')
axes[1].set_xlabel('log(1 + MedHouseValue)')
axes[1].set_ylabel('Density')
axes[1].set_title(f'Log-Transformed Price\n(skewness={log_price.skew():.2f})')
axes[1].legend()

plt.suptitle('Price Distribution: Raw vs Log-Transformed', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Log transformation makes the price distribution more symmetric.')
print('Some models perform better when the target is approximately normal.')

In [ ]:
# Geographic map of house values — full detail
fig, ax = plt.subplots(figsize=(10, 10))

# Plot cheap areas first (background), expensive areas on top
df_sorted = df.sort_values('MedHouseValue')
sc = ax.scatter(
    df_sorted['Longitude'], df_sorted['Latitude'],
    c=df_sorted['MedHouseValue'], cmap=PRICE_CMAP,
    s=2, alpha=0.7, vmin=df['MedHouseValue'].quantile(0.05), vmax=df['MedHouseValue'].quantile(0.95)
)
cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Median House Value ($100k)', fontsize=11)
ax.set_xlabel('Longitude', fontsize=12)
ax.set_ylabel('Latitude', fontsize=12)
ax.set_title('California House Prices by Location\n(Bay Area and LA Coast are the expensive clusters)', fontsize=12)
plt.tight_layout()
plt.show()

## 🔗 Section 5: Feature vs Target — What Drives Price?

Now the key question: **which features correlate with house value?**

We examine each feature against `MedHouseValue` to find signals before building any model.

### MedInc vs MedHouseValue

In [ ]:
# Sample 10% for cleaner scatter
sample = df.sample(frac=0.1, random_state=42)

r_medinc, p_medinc = stats.pearsonr(df['MedInc'], df['MedHouseValue'])
print(f'Pearson r (MedInc vs MedHouseValue): {r_medinc:.4f}  (p={p_medinc:.2e})')
print(f'r² = {r_medinc**2:.4f}  — MedInc alone explains {r_medinc**2*100:.1f}% of price variance')

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(sample['MedInc'], sample['MedHouseValue'], alpha=0.2, s=10, color='steelblue', label='Block groups')

# Regression line
m, b = np.polyfit(df['MedInc'], df['MedHouseValue'], 1)
x_line = np.linspace(df['MedInc'].min(), df['MedInc'].max(), 200)
ax.plot(x_line, m*x_line + b, 'r-', lw=2, label=f'Linear fit (r={r_medinc:.2f})')

ax.set_xlabel('Median Income ($10k)')
ax.set_ylabel('Median House Value ($100k)')
ax.set_title(f'MedInc vs MedHouseValue\nr = {r_medinc:.2f} — strong positive correlation')
ax.legend()
plt.tight_layout()
plt.show()

**📊 CONCEPT: Pearson Correlation Coefficient (r)**

The **Pearson r** measures the **linear relationship strength** between two variables:
- `r = +1`: perfect positive linear relationship
- `r = -1`: perfect negative linear relationship
- `r = 0`: no linear relationship
- `|r| > 0.5`: strong correlation
- `|r| 0.3–0.5`: moderate correlation
- `|r| < 0.3`: weak correlation

**r = 0.69** for income vs price is strong — income is our best single predictor.

In [ ]:
# HouseAge vs MedHouseValue
r_age, _ = stats.pearsonr(df['HouseAge'], df['MedHouseValue'])

# AveRooms (remove outliers for cleaner view)
df_rooms = df[df['AveRooms'] <= 20]
r_rooms, _ = stats.pearsonr(df_rooms['AveRooms'], df_rooms['MedHouseValue'])

# Population (log)
r_pop_raw, _ = stats.pearsonr(df['Population'], df['MedHouseValue'])
r_pop_log, _ = stats.pearsonr(np.log1p(df['Population']), df['MedHouseValue'])

# AveOccup (filtered)
df_occ = df[df['AveOccup'] <= 10]
r_occ, _ = stats.pearsonr(df_occ['AveOccup'], df_occ['MedHouseValue'])

print('Pearson r with MedHouseValue:')
print(f'  MedInc:             r = {r_medinc:.3f}  *** STRONGEST')
print(f'  HouseAge:           r = {r_age:.3f}')
print(f'  AveRooms (clean):   r = {r_rooms:.3f}')
print(f'  Population (raw):   r = {r_pop_raw:.3f}')
print(f'  Population (log):   r = {r_pop_log:.3f}')
print(f'  AveOccup (clean):   r = {r_occ:.3f}  (negative — crowded = cheaper)')

fig, axes = plt.subplots(2, 3, figsize=(16, 10))

pairs = [
    ('HouseAge', df, 'HouseAge', f'r = {r_age:.2f}'),
    ('AveRooms', df_rooms, 'AveRooms', f'r = {r_rooms:.2f} (outliers removed)'),
    ('Population (log)', df, None, f'r_log = {r_pop_log:.2f}'),
    ('Latitude', df.sample(frac=0.2, random_state=1), 'Latitude', 'Non-linear!'),
    ('Longitude', df.sample(frac=0.2, random_state=1), 'Longitude', 'Coastal signal'),
    ('AveOccup', df_occ.sample(frac=0.3, random_state=1), 'AveOccup', f'r = {r_occ:.2f}')
]

for ax, (title, data, xcol, note) in zip(axes.flat, pairs):
    if xcol == 'Latitude' or xcol == 'Longitude':
        sc = ax.scatter(data[xcol], data['MedHouseValue'],
                        c=data['MedHouseValue'], cmap=PRICE_CMAP, s=4, alpha=0.3)
    elif xcol is None:  # Population log
        ax.scatter(np.log1p(df['Population'].sample(2000, random_state=2)),
                   df.loc[df['Population'].sample(2000, random_state=2).index, 'MedHouseValue'],
                   alpha=0.2, s=8, color='tomato')
        xcol_label = 'log(1+Population)'
        ax.set_xlabel(xcol_label)
        ax.set_ylabel('MedHouseValue ($100k)')
        ax.set_title(f'{title} vs Price\n{note}')
        continue
    else:
        ax.scatter(data[xcol], data['MedHouseValue'], alpha=0.15, s=8, color='steelblue')
        m_, b_ = np.polyfit(data[xcol], data['MedHouseValue'], 1)
        x_ = np.linspace(data[xcol].min(), data[xcol].max(), 100)
        ax.plot(x_, m_*x_+b_, 'r-', lw=2)
    ax.set_xlabel(xcol)
    ax.set_ylabel('MedHouseValue ($100k)')
    ax.set_title(f'{title} vs Price\n{note}')

plt.suptitle('Feature vs Target: All Key Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

**Key Insight from Latitude plot:** Latitude doesn't have a simple positive or negative relationship with price. Both Southern CA (low latitude ~33°) and Northern CA Bay Area (high latitude ~37-38°) have expensive neighborhoods. This **U-shaped, non-linear relationship** is exactly why linear models struggle — they can only capture straight-line patterns.

## 🔥 Section 6: Correlation Analysis

Let's see **all relationships at once** using a correlation matrix.

In [ ]:
# Full correlation matrix
corr_matrix = df.corr()

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.zeros_like(corr_matrix, dtype=bool)  # no masking — show full matrix
sns.heatmap(
    corr_matrix,
    annot=True, fmt='.2f',
    cmap='coolwarm',
    vmin=-1, vmax=1,
    center=0,
    square=True,
    linewidths=0.5,
    ax=ax,
    cbar_kws={'shrink': 0.8}
)
ax.set_title('Pearson Correlation Matrix — California Housing', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

# Print top correlates with price
price_corr = corr_matrix['MedHouseValue'].drop('MedHouseValue').sort_values(key=abs, ascending=False)
print('\n=== CORRELATIONS WITH MedHouseValue (sorted by |r|) ===')
for feat, r in price_corr.items():
    bar = '█' * int(abs(r) * 20)
    direction = '+' if r > 0 else '-'
    print(f'  {feat:<14} {direction} {bar:<20} r={r:.3f}')

**Correlation Matrix Interpretation:**

- **MedInc** (r=0.69): Strongest predictor. Wealthy neighborhoods have higher house prices.
- **Latitude** (r=-0.14): Weak negative — not a simple linear relationship.
- **AveRooms** (r=0.15): More rooms → slightly higher prices.
- **AveOccup** (r=-0.02): Very weak — crowded households slightly lower prices.

**📊 CONCEPT: Multicollinearity**

Look at **AveRooms vs AveBedrms**: r=0.85 — very high correlation between two features!

**Multicollinearity** occurs when two features are highly correlated with *each other*. They carry redundant information — `AveRooms` and `AveBedrms` are both measuring "house size."

Problems caused by multicollinearity:
1. Makes model coefficients unstable and hard to interpret
2. The model can't tell which feature is "responsible" for the prediction
3. Can cause overfitting

Solutions: drop one of the correlated features, or use regularization (Ridge Regression).

In [ ]:
# Pair plot on a subset of key features
# Create price quantile label for coloring
df_pair = df[['MedInc', 'HouseAge', 'AveRooms', 'AveOccup', 'MedHouseValue']].copy()
df_pair = df_pair[df_pair['AveRooms'] <= 15]  # remove extreme outliers for clarity
df_pair = df_pair[df_pair['AveOccup'] <= 8]

df_pair['PriceGroup'] = pd.qcut(
    df_pair['MedHouseValue'], q=3,
    labels=['Cheap (bottom 33%)', 'Mid (33-67%)', 'Expensive (top 33%)']
)

g = sns.pairplot(
    df_pair.sample(2000, random_state=42),
    vars=['MedInc', 'HouseAge', 'AveRooms', 'AveOccup', 'MedHouseValue'],
    hue='PriceGroup',
    palette={'Cheap (bottom 33%)': '#2196F3',
             'Mid (33-67%)': '#FF9800',
             'Expensive (top 33%)': '#E91E63'},
    plot_kws={'alpha': 0.3, 's': 8},
    diag_kind='kde',
    corner=False
)
g.figure.suptitle('Pair Plot — Key Features Colored by Price Tier', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('Each cell = relationship between two variables.')
print('Diagonal = distribution of that variable.')
print('Pink dots (expensive) cluster at high MedInc — confirms income as the key driver.')

In [ ]:
# Price quintile analysis
df_q = df.copy()
df_q['price_quintile'] = pd.qcut(
    df_q['MedHouseValue'], q=5,
    labels=['Very Cheap', 'Cheap', 'Mid', 'Expensive', 'Very Expensive']
)

features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup']
quintile_means = df_q.groupby('price_quintile', observed=True)[features].mean()

# Normalize each feature to 0-1 for fair comparison
quintile_norm = (quintile_means - quintile_means.min()) / (quintile_means.max() - quintile_means.min())

fig, ax = plt.subplots(figsize=(10, 5))
sns.heatmap(
    quintile_norm.T,
    annot=quintile_means.T.round(2),
    fmt='g',
    cmap='YlOrRd',
    ax=ax,
    linewidths=0.5,
    cbar_kws={'label': 'Normalized Mean (0=lowest, 1=highest)'}
)
ax.set_title('Mean Feature Value per Price Quintile\n(cell values = actual means)', fontsize=12, fontweight='bold')
ax.set_xlabel('Price Quintile')
ax.set_ylabel('Feature')
plt.tight_layout()
plt.show()

print('Observation: MedInc consistently increases from Very Cheap → Very Expensive.')
print('This monotonic pattern confirms MedInc is the strongest predictor.')

## 🔧 Section 7: Feature Engineering

Raw features aren't always the best form for a model. **Feature engineering** means creating new, more informative features from existing ones.

Goal: give the model variables that capture the *meaning* behind the data, not just the raw numbers.

In [ ]:
# First: cap extreme outliers to prevent them from distorting the model
df_eng = df.copy()

caps = {'AveRooms': 20, 'AveBedrms': 5, 'AveOccup': 10}
for col, cap in caps.items():
    before = df_eng[col].max()
    df_eng[col] = df_eng[col].clip(upper=cap)
    after = df_eng[col].max()
    pct_affected = (df[col] > cap).sum() / len(df) * 100
    print(f'{col}: capped at {cap} (was {before:.1f}) — affected {pct_affected:.2f}% of rows')
print()
print('Capping outliers reduces the influence of extreme values on the model.')

In [ ]:
# Engineer new features

# 1. rooms_per_bedroom: spaciousness ratio
df_eng['rooms_per_bedroom'] = df_eng['AveRooms'] / df_eng['AveBedrms'].clip(lower=0.5)

# 2. income_per_room: ability to pay per room of space
df_eng['income_per_room'] = df_eng['MedInc'] / df_eng['AveRooms'].clip(lower=1)

# 3. log_population: normalize skewed population
df_eng['log_population'] = np.log1p(df_eng['Population'])

# 4. log_income: normalize skewed income
df_eng['log_income'] = np.log1p(df_eng['MedInc'])

# 5. distance_to_coast: approximate coastal distance using longitude
# CA coast runs roughly along longitude -124 to -117; approx centroid ~-119.5
df_eng['distance_to_coast'] = abs(df_eng['Longitude'] - (-119.5))

# 6. bedroom_ratio: low = more living space relative to bedrooms = nicer homes
df_eng['bedroom_ratio'] = df_eng['AveBedrms'] / df_eng['AveRooms'].clip(lower=1)

new_features = ['rooms_per_bedroom', 'income_per_room', 'log_population',
                'log_income', 'distance_to_coast', 'bedroom_ratio']

print('=== ENGINEERED FEATURES ===')
for f in new_features:
    print(f'  {f:<22}: mean={df_eng[f].mean():.3f}, std={df_eng[f].std():.3f}')

In [ ]:
# Compare correlations: original vs engineered features
original_features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

orig_corr = {f: df_eng[f].corr(df_eng['MedHouseValue']) for f in original_features}
eng_corr  = {f: df_eng[f].corr(df_eng['MedHouseValue']) for f in new_features}

all_corr = {**orig_corr, **eng_corr}
all_corr_sorted = dict(sorted(all_corr.items(), key=lambda x: abs(x[1]), reverse=True))

print('=== FEATURE CORRELATIONS WITH MedHouseValue ===')
print(f'{"Feature":<22} {"r":>8}  {"Type"}')
print('-' * 45)
for feat, r in all_corr_sorted.items():
    ftype = 'ENGINEERED' if feat in new_features else 'original'
    marker = '★' if feat in new_features else ' '
    print(f'{marker} {feat:<20} {r:>8.4f}  {ftype}')

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#e74c3c' if f in new_features else '#3498db' for f in all_corr_sorted.keys()]
bars = ax.barh(list(all_corr_sorted.keys()), [abs(v) for v in all_corr_sorted.values()],
               color=colors, edgecolor='white', alpha=0.85)
ax.set_xlabel('|Pearson r| with MedHouseValue')
ax.set_title('Feature Correlations with Price\n(red = engineered, blue = original)', fontsize=12, fontweight='bold')
ax.set_xlim(0, 0.85)

# Add value labels
for bar, (feat, r) in zip(bars, all_corr_sorted.items()):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{r:.3f}', va='center', fontsize=9)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='Engineered'),
                   Patch(facecolor='#3498db', label='Original')]
ax.legend(handles=legend_elements)
plt.tight_layout()
plt.show()

In [ ]:
# Distance to coast vs price
r_coast, _ = stats.pearsonr(df_eng['distance_to_coast'], df_eng['MedHouseValue'])
print(f'Correlation: distance_to_coast vs MedHouseValue: r = {r_coast:.4f}')
print('Negative r confirms: farther from coast = lower price')
print()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sample2 = df_eng.sample(3000, random_state=42)
axes[0].scatter(sample2['distance_to_coast'], sample2['MedHouseValue'],
                alpha=0.2, s=8, color='darkcyan')
m_, b_ = np.polyfit(df_eng['distance_to_coast'], df_eng['MedHouseValue'], 1)
x_ = np.linspace(0, df_eng['distance_to_coast'].max(), 100)
axes[0].plot(x_, m_*x_+b_, 'r-', lw=2, label=f'r={r_coast:.2f}')
axes[0].set_xlabel('Distance from Coast (longitude degrees)')
axes[0].set_ylabel('Median House Value ($100k)')
axes[0].set_title('Distance to Coast vs Price')
axes[0].legend()

# rooms_per_bedroom vs price
df_rpb = df_eng[df_eng['rooms_per_bedroom'] <= 20]
r_rpb, _ = stats.pearsonr(df_rpb['rooms_per_bedroom'], df_rpb['MedHouseValue'])
sample3 = df_rpb.sample(3000, random_state=42)
axes[1].scatter(sample3['rooms_per_bedroom'], sample3['MedHouseValue'],
                alpha=0.2, s=8, color='darkorchid')
m_, b_ = np.polyfit(df_rpb['rooms_per_bedroom'], df_rpb['MedHouseValue'], 1)
x_ = np.linspace(df_rpb['rooms_per_bedroom'].min(), 20, 100)
axes[1].plot(x_, m_*x_+b_, 'r-', lw=2, label=f'r={r_rpb:.2f}')
axes[1].set_xlabel('Rooms per Bedroom (spaciousness)')
axes[1].set_ylabel('Median House Value ($100k)')
axes[1].set_title('Rooms per Bedroom vs Price')
axes[1].legend()

plt.suptitle('Engineered Features vs Price', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Feature Engineering Results:**
- `income_per_room` can have higher correlation with price than raw income alone
- `distance_to_coast` captures the coastal premium that latitude/longitude individually miss
- `rooms_per_bedroom` measures spaciousness — a quality indicator raw room counts don't capture
- `log_population` and `log_income` reduce skewness, making these features more model-friendly

## 🤖 Section 8: Building the Prediction

We've explored, understood, and engineered our data. Now: **can we actually predict house prices?**

We'll build 4 models, compare their performance, and understand *why* some work better than others.

In [ ]:
# Feature set: original (capped) + engineered features
all_features = [
    'MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population',
    'AveOccup', 'Latitude', 'Longitude',
    'rooms_per_bedroom', 'income_per_room', 'log_population',
    'log_income', 'distance_to_coast', 'bedroom_ratio'
]

X = df_eng[all_features]
y = df_eng['MedHouseValue']

# Train/test split: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Normalize features (StandardScaler: mean=0, std=1)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)  # use train stats on test!

print(f'Training set: {X_train.shape[0]:,} rows')
print(f'Test set:     {X_test.shape[0]:,} rows')
print(f'Features:     {X_train.shape[1]}')
print()
print('StandardScaler: transforms each feature to have mean=0, std=1')
print('IMPORTANT: fit on TRAIN only, then transform both train and test.')
print('           Never fit on test — that would leak future information!')

In [ ]:
# Build and evaluate 4 models
models = {
    'Linear Regression': LinearRegression(),
    'Ridge Regression': Ridge(alpha=1.0),
    'Decision Tree (depth=8)': DecisionTreeRegressor(max_depth=8, random_state=42),
    'Random Forest (50 trees)': RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
}

results = {}
predictions = {}

for name, model in models.items():
    # Tree models don't need scaled features, but it doesn't hurt
    if 'Forest' in name or 'Tree' in name:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
    else:
        model.fit(X_train_scaled, y_train)
        y_pred = model.predict(X_test_scaled)

    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    results[name] = {'MAE': mae, 'RMSE': rmse, 'R²': r2}
    predictions[name] = y_pred
    print(f'{name}:')
    print(f'  MAE  = ${mae*100:.1f}k  (avg prediction error)')
    print(f'  RMSE = ${rmse*100:.1f}k  (penalizes large errors more)')
    print(f'  R²   = {r2:.4f}   (proportion of variance explained)')
    print()

# Summary table
results_df = pd.DataFrame(results).T
results_df = results_df.sort_values('R²', ascending=False)
print('=== MODEL COMPARISON TABLE ===')
print(results_df.round(4))

**📊 CONCEPT: Regression Metrics**

| Metric | Formula | Meaning |
|--------|---------|----------|
| **MAE** | mean(|actual - predicted|) | Average error in same units as target |
| **RMSE** | √mean((actual - predicted)²) | Like MAE but penalizes large errors more |
| **R²** | 1 - SS_res/SS_tot | % of variance explained (0=terrible, 1=perfect) |

**R² interpretation:** If R²=0.81, the model explains 81% of the variation in house prices. The remaining 19% is unexplained — noise or missing features.

**Why Random Forest outperforms Linear Regression:** Random Forests can model *non-linear* relationships (like the U-shaped latitude pattern we saw) and *interactions* between features (coastal + high-income = extra expensive).

In [ ]:
# Analyze the best model (Random Forest)
best_model_name = results_df.index[0]
y_pred_best = predictions[best_model_name]
residuals = y_test - y_pred_best

print(f'Best model: {best_model_name}')
print(f'R² = {results_df.loc[best_model_name, "R²"]:.4f}')
print(f'MAE = ${results_df.loc[best_model_name, "MAE"]*100:.1f}k')
print()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Plot 1: Predicted vs Actual
error_magnitude = np.abs(residuals)
sc = axes[0].scatter(y_test, y_pred_best, c=error_magnitude, cmap='RdYlGn_r',
                     s=5, alpha=0.4, vmin=0, vmax=1)
# Perfect prediction line
lims = [min(y_test.min(), y_pred_best.min()), max(y_test.max(), y_pred_best.max())]
axes[0].plot(lims, lims, 'k--', lw=2, label='Perfect prediction')
plt.colorbar(sc, ax=axes[0], label='|Error| ($100k)')
axes[0].set_xlabel('Actual Price ($100k)')
axes[0].set_ylabel('Predicted Price ($100k)')
axes[0].set_title(f'Predicted vs Actual\n({best_model_name})')
axes[0].legend(fontsize=9)

# Plot 2: Residuals histogram
axes[1].hist(residuals, bins=60, color='steelblue', edgecolor='white', density=True, alpha=0.8)
kde_x = np.linspace(residuals.min(), residuals.max(), 300)
kde = stats.gaussian_kde(residuals)
axes[1].plot(kde_x, kde(kde_x), 'r-', lw=2, label='KDE')
axes[1].axvline(0, color='black', lw=2, linestyle='--', label='Zero error')
axes[1].axvline(residuals.mean(), color='orange', lw=2, linestyle=':',
                label=f'Mean error={residuals.mean():.3f}')
axes[1].set_xlabel('Residual (Actual - Predicted) $100k')
axes[1].set_ylabel('Density')
axes[1].set_title('Residuals Distribution\n(should be ~normal centered at 0)')
axes[1].legend(fontsize=9)

# Plot 3: Residuals vs Predicted (check for heteroscedasticity)
axes[2].scatter(y_pred_best, residuals, alpha=0.2, s=5, color='darkorange')
axes[2].axhline(0, color='red', lw=2, linestyle='--')
axes[2].axhline(residuals.std(), color='gray', lw=1, linestyle=':', label=f'+1σ = {residuals.std():.2f}')
axes[2].axhline(-residuals.std(), color='gray', lw=1, linestyle=':', label=f'-1σ')
axes[2].set_xlabel('Predicted Price ($100k)')
axes[2].set_ylabel('Residual')
axes[2].set_title('Residuals vs Predicted\n(check for patterns / heteroscedasticity)')
axes[2].legend(fontsize=9)

plt.suptitle(f'{best_model_name} — Model Diagnostics', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print()
print('Residual skewness:', round(stats.skew(residuals), 3))
print('If residuals are approximately normal and centered at 0 — the model is well-calibrated.')
print('If there are patterns in the Residuals vs Predicted plot — the model is missing something.')

**📊 CONCEPT: Heteroscedasticity**

**Heteroscedasticity** = the variance of residuals changes with predicted value.

In the Residuals vs Predicted plot:
- **Good:** residuals are randomly scattered around 0, no pattern
- **Bad:** residuals fan out (larger errors for higher prices) — model struggles at extremes

The $500k cap in our data artificially compresses errors at the high end. This is a known limitation of the dataset.

In [ ]:
# Random Forest feature importances
rf_model = models['Random Forest (50 trees)']
importances = rf_model.feature_importances_
feat_imp_df = pd.DataFrame({
    'feature': all_features,
    'importance': importances
}).sort_values('importance', ascending=False)

print('=== RANDOM FOREST FEATURE IMPORTANCES ===')
for _, row in feat_imp_df.iterrows():
    bar = '█' * int(row['importance'] * 200)
    print(f'  {row["feature"]:<22}: {bar:<20} {row["importance"]:.4f}')

fig, ax = plt.subplots(figsize=(10, 7))
colors_imp = ['#e74c3c' if f in new_features else '#3498db' for f in feat_imp_df['feature']]
bars = ax.barh(feat_imp_df['feature'], feat_imp_df['importance'],
               color=colors_imp, edgecolor='white', alpha=0.85)

# Add value labels
for bar, val in zip(bars, feat_imp_df['importance']):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='Engineered feature'),
                   Patch(facecolor='#3498db', label='Original feature')]
ax.legend(handles=legend_elements)
ax.set_xlabel('Feature Importance (Gini impurity reduction)')
ax.set_title('Random Forest Feature Importances\n(confirms EDA findings)', fontsize=12, fontweight='bold')
ax.set_xlim(0, feat_imp_df['importance'].max() * 1.2)
plt.tight_layout()
plt.show()

top_feat = feat_imp_df.iloc[0]['feature']
print(f'\nTop feature: {top_feat}')
print('Does this match our EDA finding? (MedInc had r=0.69 with price)')

In [ ]:
# Geographic prediction map: actual vs predicted
# Get test set indices and their lat/long
test_geo = df_eng.iloc[X_test.index][['Latitude', 'Longitude']].copy()
test_geo['actual'] = y_test.values
test_geo['predicted'] = y_pred_best

vmin = df_eng['MedHouseValue'].quantile(0.05)
vmax = df_eng['MedHouseValue'].quantile(0.95)

fig, axes = plt.subplots(1, 2, figsize=(16, 9))

for ax, col, title in zip(axes,
                           ['actual', 'predicted'],
                           ['Actual Prices', 'Predicted Prices (Random Forest)']):
    sc = ax.scatter(
        test_geo['Longitude'], test_geo['Latitude'],
        c=test_geo[col], cmap=PRICE_CMAP,
        s=2, alpha=0.6, vmin=vmin, vmax=vmax
    )
    cbar = plt.colorbar(sc, ax=ax)
    cbar.set_label('Median House Value ($100k)', fontsize=10)
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_facecolor('#0d0d1a')

plt.suptitle('Does the Model Capture the Geographic Price Pattern?', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('If the model learned well, the two maps should look similar.')
print('Key check: Does it capture the Bay Area and LA coastal premium?')

## 📚 Section 9: What We Learned

We started with raw data about California neighborhoods and ended with a model that can predict house prices. Along the way, every statistical concept served a purpose.

### Key Findings Summary

| Concept | Used For | Key Finding |
|---------|----------|-------------|
| Descriptive Statistics | Understanding each feature | Population is highly skewed; MedInc has moderate skew |
| Mean vs Median | Comparing central tendency | Right-skewed income: mean > median; median is more robust |
| Skewness | Detecting non-normal distributions | Population skewness >5 — needs log transform |
| Log Transformation | Normalizing skewed features | Reduces skewness from >5 to ~0.3 for Population |
| Outliers (IQR method) | Cleaning AveRooms, AveOccup | ~1-2% of rows had extreme values; capped before modeling |
| Scatter Plot | Feature vs target relationships | MedInc shows clear linear relationship with price |
| Pearson r | Quantifying linear correlation | MedInc r=0.69 — strongest single predictor |
| Correlation Matrix | All relationships at once | AveRooms/AveBedrms highly correlated (multicollinearity) |
| Pair Plot | Visual multi-feature exploration | Expensive homes cluster at high MedInc across all features |
| Spatial Visualization | Geographic patterns | Coastal clusters in Bay Area and LA Basin clear in price maps |
| Feature Engineering | Creating better predictors | income_per_room, distance_to_coast improve correlations |
| Train/Test Split | Honest model evaluation | 80/20 split prevents overfitting measurement |
| StandardScaler | Feature normalization | Required for linear models; coefficients comparable |
| MAE/RMSE/R² | Model performance metrics | Random Forest R²≈0.81 — explains 81% of price variance |
| Residual Analysis | Model diagnostic | Approximately normal residuals = well-calibrated model |
| Feature Importance | Understanding model decisions | MedInc is most important feature — confirms EDA |

### The Big Insights

1. **Income is king.** Median income of a neighborhood explains the majority of price variation — r=0.69 is a remarkably strong single-feature correlation.

2. **Geography matters.** The map reveals what numbers alone miss: spatial clustering of prices along the California coast. The Bay Area and LA Basin command a significant premium.

3. **Non-linearity is real.** Latitude has a non-linear (roughly U-shaped) relationship with price. This is why Random Forests outperform Linear Regression here — they capture these patterns.

4. **Feature engineering creates value.** `income_per_room` and `distance_to_coast` capture aspects of price that raw features alone miss.

5. **The data has limitations.** The $500k cap in the dataset means we likely underestimate prices in the most expensive neighborhoods — a reminder that **data quality shapes model quality**.

6. **EDA and modeling agree.** The features our exploratory analysis identified as important (MedInc, geography) are the same features the Random Forest assigns highest importance to. This is a good sign — the data tells a consistent story.

In [ ]:
# Final model comparison visualization
results_sorted = results_df.sort_values('R²')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, metric, color, label in zip(
    axes,
    ['MAE', 'RMSE', 'R²'],
    ['#e74c3c', '#e67e22', '#27ae60'],
    ['MAE ($100k) — lower is better', 'RMSE ($100k) — lower is better', 'R² — higher is better']
):
    vals = results_sorted[metric]
    ax.barh(results_sorted.index, vals, color=color, edgecolor='white', alpha=0.85)
    for i, (idx, val) in enumerate(vals.items()):
        ax.text(val * 0.02, i, f'{val:.3f}', va='center', fontsize=9, color='white', fontweight='bold')
    ax.set_xlabel(label)
    ax.set_title(metric)

plt.suptitle('Final Model Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print('='*60)
print('FINAL RESULTS SUMMARY')
print('='*60)
for model_name, row in results_df.iterrows():
    print(f"{model_name}")
    print(f"  MAE=${row['MAE']*100:.1f}k | RMSE=${row['RMSE']*100:.1f}k | R²={row['R²']:.4f}")
print()
print(f'Winner: {results_df.index[0]}')
print(f'It explains {results_df.iloc[0]["R²"]*100:.1f}% of the variation in California house prices.')
print(f'Average prediction error: ${results_df.iloc[0]["MAE"]*100:.1f}k')